# Smart Greenhouse Direct Multi-Horizon GRU/LSTM Baselines

## 00 - Experiment Overview

Controlled experiment: giu lookback/model capacity/training protocol cua one-step baseline, nhung truc tiep
du bao cac horizon `+1,+3,+6,+12,+24h`. Model development chi dung TRAIN 2018-2023 va VALIDATION 2024.

> Notebook nay khong tao va khong evaluate temporal, held-out scenario hay combined final-test loaders.

## 01 - Configuration

Centralized scientific defaults. Local smoke override bang environment; committed default van la full Colab mode.

In [ ]:
import os
from pathlib import Path

SEED = 20260816
LOOKBACK_STEPS = 24
FORECAST_HORIZONS = (1, 3, 6, 12, 24)
MAX_FORECAST_HORIZON = max(FORECAST_HORIZONS)
BATCH_SIZE = 256
MAX_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 7
GRADIENT_CLIP_NORM = 1.0
HIDDEN_SIZE = 64
NUM_LAYERS = 1
NUM_WORKERS = 2

MULTI_HORIZON_SMOKE_TEST = False
MULTI_HORIZON_SMOKE_TEST = os.getenv(
    "GREENHOUSE_MULTI_HORIZON_SMOKE_TEST", str(MULTI_HORIZON_SMOKE_TEST)
).lower() in {"1", "true", "yes"}
SMOKE_MAX_EPOCHS = 1
SMOKE_MAX_TRAIN_BATCHES = 3
SMOKE_MAX_VALIDATION_BATCHES = 2

default_root = Path("/content/smart_greenhouse_dataset") if Path("/content").exists() else Path.cwd()
DATA_ROOT = Path(os.getenv("GREENHOUSE_DATA_ROOT", str(default_root))).expanduser()
PREPROCESSING_ARTIFACT_DIR = Path(os.getenv(
    "GREENHOUSE_PREPROCESSING_ARTIFACT_DIR", str(DATA_ROOT / "artifacts" / "preprocessing")
))
MULTI_HORIZON_ARTIFACT_DIR = Path(os.getenv(
    "GREENHOUSE_MULTI_HORIZON_ARTIFACT_DIR",
    str(DATA_ROOT / "artifacts" / (
        "multi_horizon_training_smoke" if MULTI_HORIZON_SMOKE_TEST else "multi_horizon_training"
    )),
))
INDEX_FILE = DATA_ROOT / "full_dataset_index.csv"

EXPECTED_SCENARIOS = 24
EXPECTED_ROWS_PER_SCENARIO = 70_128
EXPECTED_TOTAL_ROWS = 1_683_072
EXPECTED_TRAIN_WINDOWS = 1_050_740
EXPECTED_VALIDATION_WINDOWS = 175_220
print(f"DATA_ROOT={DATA_ROOT.resolve()}")
print(f"MULTI_HORIZON_SMOKE_TEST={MULTI_HORIZON_SMOKE_TEST}")

## 02 - Imports

Standard Colab packages only; no CUDA reinstall and no dependency on Notebook 02 checkpoints.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
import json
import math
import platform
import random
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler

## 03 - Reproducibility

Seed Python, NumPy, PyTorch va DataLoader; recurrent baselines duoc khoi tao/train theo cung seed policy.

In [ ]:
def set_reproducibility(seed: int) -> torch.Generator:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


data_loader_generator = set_reproducibility(SEED)

## 04 - Device

Full training requires Colab CUDA; CPU chi duoc phep cho bounded local smoke.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
elif not MULTI_HORIZON_SMOKE_TEST:
    raise RuntimeError("Full multi-horizon training requires a CUDA-enabled Colab runtime")

## 05 - Paths

Normalize indexed Windows paths for Linux. Membership is never discovered by glob.

In [ ]:
def normalize_index_path(raw_path: str) -> Path:
    normalized = str(raw_path).strip().replace("\\", "/")
    if not normalized:
        raise ValueError("Empty indexed path")
    return Path(normalized)


def resolve_scenario_path(raw_path: str, data_root: Path) -> Path:
    normalized = normalize_index_path(raw_path)
    candidates = [
        normalized if normalized.is_absolute() else data_root / normalized,
        data_root / "outputs" / "full_generation" / "ml" / normalized.name,
        data_root / "ml" / normalized.name,
        data_root / normalized.name,
    ]
    unique = list(dict.fromkeys(candidate.resolve() for candidate in candidates))
    existing = [candidate for candidate in unique if candidate.is_file()]
    if unique[0] in existing:
        return unique[0]
    if len(existing) != 1:
        raise FileNotFoundError(f"Cannot uniquely resolve {raw_path!r}: {existing}")
    warnings.warn(f"Explicit path fallback used for {raw_path!r}: {existing[0]}")
    return existing[0]

## 06 - Load Preprocessing Artifacts

Load-only scalers/split/config. Full mode rejects smoke-fitted preprocessing artifacts.

In [ ]:
preprocessing_paths = {
    "feature_scaler": PREPROCESSING_ARTIFACT_DIR / "feature_scaler.pkl",
    "target_scaler": PREPROCESSING_ARTIFACT_DIR / "target_scaler.pkl",
    "split_manifest": PREPROCESSING_ARTIFACT_DIR / "split_manifest.json",
    "preprocessing_config": PREPROCESSING_ARTIFACT_DIR / "preprocessing_config.json",
}
missing = [str(path) for path in preprocessing_paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Locked preprocessing artifacts missing: {missing}")
feature_scaler = joblib.load(preprocessing_paths["feature_scaler"])
target_scaler = joblib.load(preprocessing_paths["target_scaler"])
split_manifest = json.loads(preprocessing_paths["split_manifest"].read_text(encoding="utf-8"))
preprocessing_config = json.loads(
    preprocessing_paths["preprocessing_config"].read_text(encoding="utf-8")
)
if not MULTI_HORIZON_SMOKE_TEST and preprocessing_config.get("smoke_test_execution"):
    raise ValueError("Full training refuses smoke-fitted preprocessing artifacts")
for scaler in (feature_scaler, target_scaler):
    if not all(hasattr(scaler, name) for name in ("mean_", "scale_", "transform")):
        raise TypeError("A fitted StandardScaler-compatible artifact is required")
print(f"Loaded preprocessing artifacts from {PREPROCESSING_ARTIFACT_DIR.resolve()}")

## 07 - Validate Locked Contract

Immutable feature/target/horizon order, split IDs va date ranges. Held-out IDs chi duoc audit provenance.

In [ ]:
FEATURE_COLUMNS = [
    "air_temperature", "air_humidity", "soil_temperature", "soil_moisture",
    "light_lux", "pump_state", "fan_state", "grow_light_state",
]
TARGET_COLUMNS = [
    "air_temperature", "air_humidity", "soil_temperature", "soil_moisture", "light_lux"
]
BINARY_FEATURE_COLUMNS = ["pump_state", "fan_state", "grow_light_state"]
SOURCE_COLUMNS = ["timestamp", *FEATURE_COLUMNS]
if preprocessing_config["feature_columns"] != FEATURE_COLUMNS:
    raise ValueError("Locked feature order changed")
if preprocessing_config["target_columns"] != TARGET_COLUMNS:
    raise ValueError("Locked target order changed")
if int(split_manifest["lookback_steps"]) != LOOKBACK_STEPS:
    raise ValueError("Locked lookback changed")
if FORECAST_HORIZONS != (1, 3, 6, 12, 24):
    raise ValueError("Forecast horizon order changed")

development_scenario_ids = list(split_manifest["development_scenario_ids"])
held_out_scenario_ids = list(split_manifest["held_out_scenario_ids"])
if len(development_scenario_ids) != 20 or len(held_out_scenario_ids) != 4:
    raise ValueError("Locked split must remain 20 development / 4 held-out")
if set(development_scenario_ids).intersection(held_out_scenario_ids):
    raise ValueError("Development and held-out scenarios overlap")
train_start, train_end = map(pd.Timestamp, split_manifest["train_date_range"])
validation_start, validation_end = map(pd.Timestamp, split_manifest["validation_date_range"])
if (train_start, train_end) != (
    pd.Timestamp("2018-01-01 00:00"), pd.Timestamp("2023-12-31 23:00")
):
    raise ValueError("TRAIN range changed")
if (validation_start, validation_end) != (
    pd.Timestamp("2024-01-01 00:00"), pd.Timestamp("2024-12-31 23:00")
):
    raise ValueError("Validation range changed")
print("Locked split and direct multi-horizon contract PASS")

## 08 - Load Canonical Index

Index la membership authority. Tat ca 24 IDs phai duoc split manifest partition chinh xac.

In [ ]:
def load_canonical_index(path: Path) -> pd.DataFrame:
    index = pd.read_csv(path)
    required = {"parameter_set_id", "ml_file", "ml_rows", "config_hash", "validation_status"}
    if required.difference(index.columns):
        raise ValueError("Canonical index schema mismatch")
    if len(index) != EXPECTED_SCENARIOS:
        raise ValueError("Canonical scenario count mismatch")
    if index["parameter_set_id"].duplicated().any() or index["config_hash"].duplicated().any():
        raise ValueError("Duplicate canonical identity/config")
    if not (index["ml_rows"].astype(int) == EXPECTED_ROWS_PER_SCENARIO).all():
        raise ValueError("Canonical row-count mismatch")
    if not (index["validation_status"] == "PASS").all():
        raise ValueError("Non-PASS canonical scenario")
    if set(index["parameter_set_id"]) != set(development_scenario_ids + held_out_scenario_ids):
        raise ValueError("Split IDs do not partition canonical index")
    return index.sort_values("parameter_set_id").reset_index(drop=True)


canonical_index = load_canonical_index(INDEX_FILE)
assert int(canonical_index["ml_rows"].astype(int).sum()) == EXPECTED_TOTAL_ROWS
print(f"Canonical index PASS: {len(canonical_index)} scenarios")

## 09 - Resolve Development Scenario Files

Chi resolve/load 20 development scenarios. Held-out CSV paths khong duoc resolve thanh training inputs.

In [ ]:
index_by_id = canonical_index.set_index("parameter_set_id")
development_scenario_paths = {
    scenario_id: resolve_scenario_path(index_by_id.loc[scenario_id, "ml_file"], DATA_ROOT)
    for scenario_id in development_scenario_ids
}
if set(development_scenario_paths) != set(development_scenario_ids):
    raise AssertionError("Development path resolution mismatch")
print(f"Resolved development-only files: {len(development_scenario_paths)}")

## 10 - Build Per-Scenario Arrays

Validate full source, scale each timestep once, cache contiguous arrays including raw actuators for post-hoc audit.

In [ ]:
@dataclass(frozen=True)
class ScenarioArrays:
    timestamps: np.ndarray
    scaled_features: np.ndarray
    scaled_targets: np.ndarray
    raw_targets: np.ndarray
    raw_actuators: np.ndarray


def validate_source_frame(frame: pd.DataFrame, scenario_id: str) -> pd.DataFrame:
    if list(frame.columns) != SOURCE_COLUMNS or len(frame) != EXPECTED_ROWS_PER_SCENARIO:
        raise ValueError(f"{scenario_id}: schema/row mismatch")
    frame = frame.copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], errors="raise")
    if frame["timestamp"].duplicated().any() or not frame["timestamp"].is_monotonic_increasing:
        raise ValueError(f"{scenario_id}: invalid timestamp order")
    if not (frame["timestamp"].diff().dropna() == pd.Timedelta(hours=1)).all():
        raise ValueError(f"{scenario_id}: non-hourly gap")
    numeric = frame[FEATURE_COLUMNS].to_numpy(np.float64)
    if frame[FEATURE_COLUMNS].isna().any().any() or not np.isfinite(numeric).all():
        raise ValueError(f"{scenario_id}: NaN/Inf")
    for column in BINARY_FEATURE_COLUMNS:
        if not set(frame[column].unique()).issubset({0, 1}):
            raise ValueError(f"{scenario_id}: nonbinary actuator")
    return frame


def frame_to_arrays(frame: pd.DataFrame) -> ScenarioArrays:
    raw_targets = frame[TARGET_COLUMNS].to_numpy(np.float64)
    raw_actuators = frame[BINARY_FEATURE_COLUMNS].to_numpy(np.float32)
    scaled_sensors = feature_scaler.transform(raw_targets).astype(np.float32)
    return ScenarioArrays(
        timestamps=np.ascontiguousarray(frame["timestamp"].to_numpy(dtype="datetime64[ns]")),
        scaled_features=np.ascontiguousarray(np.column_stack([scaled_sensors, raw_actuators])),
        scaled_targets=np.ascontiguousarray(
            target_scaler.transform(raw_targets).astype(np.float32)
        ),
        raw_targets=np.ascontiguousarray(raw_targets.astype(np.float32)),
        raw_actuators=np.ascontiguousarray(raw_actuators),
    )


active_development_ids = development_scenario_ids[:1] if MULTI_HORIZON_SMOKE_TEST else development_scenario_ids
scenario_arrays: dict[str, ScenarioArrays] = {}
for scenario_id in active_development_ids:
    source = validate_source_frame(pd.read_csv(development_scenario_paths[scenario_id]), scenario_id)
    scenario_arrays[scenario_id] = frame_to_arrays(source)
print(f"Cached arrays for {len(scenario_arrays)} development scenarios")

## 11 - Define Multi-Horizon Window Semantics

Compact index stores input end `t`; all five target positions are `t + horizon`. Inputs end before every target.

In [ ]:
@dataclass(frozen=True)
class MultiHorizonIndex:
    split_name: str
    scenario_ids: tuple[str, ...]
    scenario_codes: np.ndarray
    input_end_positions: np.ndarray
    target_start: pd.Timestamp
    target_end: pd.Timestamp

    def __len__(self) -> int:
        return int(len(self.input_end_positions))

    def resolve(self, item: int) -> tuple[str, int]:
        return self.scenario_ids[int(self.scenario_codes[item])], int(self.input_end_positions[item])


def valid_input_end_positions(
    arrays: ScenarioArrays,
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
) -> np.ndarray:
    timestamps = arrays.timestamps
    candidates = np.arange(LOOKBACK_STEPS - 1, len(timestamps) - MAX_FORECAST_HORIZON, dtype=np.int64)
    input_start = candidates - LOOKBACK_STEPS + 1
    target_matrix = candidates[:, None] + np.asarray(FORECAST_HORIZONS, dtype=np.int64)[None, :]
    targets = timestamps[target_matrix]
    valid_targets = (
        (targets >= np.datetime64(target_start))
        & (targets <= np.datetime64(target_end))
    ).all(axis=1)
    continuous_input = (
        timestamps[candidates] - timestamps[input_start]
        == np.timedelta64(LOOKBACK_STEPS - 1, "h")
    )
    continuous_reach = (
        timestamps[candidates + MAX_FORECAST_HORIZON] - timestamps[candidates]
        == np.timedelta64(MAX_FORECAST_HORIZON, "h")
    )
    return candidates[valid_targets & continuous_input & continuous_reach].astype(np.int32)


def build_multi_horizon_index(
    arrays_by_scenario: dict[str, ScenarioArrays],
    scenario_ids: list[str],
    split_name: str,
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
) -> MultiHorizonIndex:
    ordered = tuple(sorted(scenario_ids))
    codes, positions = [], []
    for code_value, scenario_id in enumerate(ordered):
        scenario_positions = valid_input_end_positions(
            arrays_by_scenario[scenario_id], target_start, target_end
        )
        codes.append(np.full(len(scenario_positions), code_value, dtype=np.int16))
        positions.append(scenario_positions)
    return MultiHorizonIndex(
        split_name=split_name,
        scenario_ids=ordered,
        scenario_codes=np.concatenate(codes),
        input_end_positions=np.concatenate(positions),
        target_start=target_start,
        target_end=target_end,
    )

## 12 - Build Train/Validation Sequence Indices

Chi hai indices duoc tao. Full counts duoc derive va assert; validation targets deu nam trong 2024.

In [ ]:
if MULTI_HORIZON_SMOKE_TEST:
    active_train_start = pd.Timestamp("2018-01-01 00:00")
    active_train_end = pd.Timestamp("2018-01-14 23:00")
    active_validation_start = pd.Timestamp("2024-01-01 00:00")
    active_validation_end = pd.Timestamp("2024-01-07 23:00")
else:
    active_train_start, active_train_end = train_start, train_end
    active_validation_start, active_validation_end = validation_start, validation_end

train_sequence_index = build_multi_horizon_index(
    scenario_arrays, active_development_ids, "train", active_train_start, active_train_end
)
validation_sequence_index = build_multi_horizon_index(
    scenario_arrays, active_development_ids, "validation", active_validation_start, active_validation_end
)
actual_window_counts = {
    "train": len(train_sequence_index),
    "validation": len(validation_sequence_index),
}
if not MULTI_HORIZON_SMOKE_TEST and actual_window_counts != {
    "train": EXPECTED_TRAIN_WINDOWS,
    "validation": EXPECTED_VALIDATION_WINDOWS,
}:
    raise ValueError(f"Multi-horizon window-count mismatch: {actual_window_counts}")
derived_train_per_scenario = 52_537
derived_validation_per_scenario = 8_761
assert derived_train_per_scenario * len(development_scenario_ids) == EXPECTED_TRAIN_WINDOWS
assert derived_validation_per_scenario * len(development_scenario_ids) == EXPECTED_VALIDATION_WINDOWS
print(f"Train/validation-only window counts: {actual_window_counts}")

## 13 - Dataset/DataLoader

Lazy slice `[24,8]`; targets resolve `[t+1,t+3,t+6,t+12,t+24]` thanh `[5,5]`. No final-test loader exists.

In [ ]:
class MultiHorizonDataset(Dataset):
    def __init__(self, arrays: dict[str, ScenarioArrays], index: MultiHorizonIndex) -> None:
        self.arrays = arrays
        self.index = index
        self.horizon_offsets = np.asarray(FORECAST_HORIZONS, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.index)

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor]:
        scenario_id, input_end = self.index.resolve(item)
        arrays = self.arrays[scenario_id]
        input_start = input_end - LOOKBACK_STEPS + 1
        features = arrays.scaled_features[input_start : input_end + 1]
        targets = arrays.scaled_targets[input_end + self.horizon_offsets]
        if features.shape != (LOOKBACK_STEPS, len(FEATURE_COLUMNS)):
            raise RuntimeError("Invalid input shape")
        if targets.shape != (len(FORECAST_HORIZONS), len(TARGET_COLUMNS)):
            raise RuntimeError("Invalid multi-horizon target shape")
        return torch.from_numpy(features), torch.from_numpy(targets)


train_dataset = MultiHorizonDataset(scenario_arrays, train_sequence_index)
validation_dataset = MultiHorizonDataset(scenario_arrays, validation_sequence_index)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


effective_workers = 0 if MULTI_HORIZON_SMOKE_TEST else NUM_WORKERS
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=effective_workers,
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=effective_workers > 0,
    worker_init_fn=seed_worker if effective_workers else None,
    generator=data_loader_generator,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=effective_workers,
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=effective_workers > 0,
    worker_init_fn=seed_worker if effective_workers else None,
)
assert isinstance(train_loader.sampler, RandomSampler)
assert isinstance(validation_loader.sampler, SequentialSampler)

## 14 - Validate Shapes and Numerical Equivalence

Assert horizon/target order, past-only inputs, validation boundary context va direct-scaler equivalence.

In [ ]:
sample_features, sample_targets = next(iter(train_loader))
assert sample_features.shape[1:] == (24, 8)
assert sample_targets.shape[1:] == (5, 5)
assert sample_features.dtype == torch.float32 and sample_targets.dtype == torch.float32
assert torch.isfinite(sample_features).all() and torch.isfinite(sample_targets).all()


def assert_reference_semantics(index: MultiHorizonIndex, item: int) -> None:
    scenario_id, input_end = index.resolve(item)
    arrays = scenario_arrays[scenario_id]
    input_start = input_end - LOOKBACK_STEPS + 1
    target_positions = input_end + np.asarray(FORECAST_HORIZONS)
    assert input_start >= 0 and target_positions.min() > input_end
    target_timestamps = arrays.timestamps[target_positions]
    assert (target_timestamps >= np.datetime64(index.target_start)).all()
    assert (target_timestamps <= np.datetime64(index.target_end)).all()
    actual_features, actual_targets = (
        train_dataset[item] if index.split_name == "train" else validation_dataset[item]
    )
    expected_targets = target_scaler.transform(
        arrays.raw_targets[target_positions].astype(np.float64)
    ).astype(np.float32)
    np.testing.assert_allclose(actual_targets.numpy(), expected_targets, rtol=1e-6, atol=1e-6)
    np.testing.assert_array_equal(
        actual_features.numpy()[:, -3:], arrays.raw_actuators[input_start : input_end + 1]
    )


for index in (train_sequence_index, validation_sequence_index):
    for item in (0, len(index) // 2, -1):
        assert_reference_semantics(index, item)
boundary_scenario, boundary_input_end = validation_sequence_index.resolve(0)
boundary_targets = scenario_arrays[boundary_scenario].timestamps[
    boundary_input_end + np.asarray(FORECAST_HORIZONS)
]
assert boundary_targets[0] == np.datetime64(active_validation_start)
assert scenario_arrays[boundary_scenario].timestamps[boundary_input_end] < boundary_targets[0]
print(f"Shape/leakage/equivalence PASS: X={tuple(sample_features.shape)}, Y={tuple(sample_targets.shape)}")

## 15 - Last-Value Persistence

Du bao moi horizon bang raw sensor state tai input end `t`.

In [ ]:
def baseline_validation_arrays(
    index: MultiHorizonIndex,
    arrays_by_scenario: dict[str, ScenarioArrays],
    max_samples: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    count = len(index) if max_samples is None else min(len(index), max_samples)
    last_value_predictions, seasonal_predictions, targets = [], [], []
    horizons = np.asarray(FORECAST_HORIZONS, dtype=np.int64)
    for item in range(count):
        scenario_id, input_end = index.resolve(item)
        arrays = arrays_by_scenario[scenario_id]
        target_positions = input_end + horizons
        last_value_predictions.append(np.repeat(arrays.raw_targets[[input_end]], len(horizons), axis=0))
        seasonal_predictions.append(arrays.raw_targets[input_end + horizons - 24])
        targets.append(arrays.raw_targets[target_positions])
    return (
        np.asarray(last_value_predictions, np.float32),
        np.asarray(seasonal_predictions, np.float32),
        np.asarray(targets, np.float32),
    )


max_validation_samples = (
    BATCH_SIZE * SMOKE_MAX_VALIDATION_BATCHES if MULTI_HORIZON_SMOKE_TEST else None
)
last_value_raw, seasonal_raw, validation_targets_raw = baseline_validation_arrays(
    validation_sequence_index, scenario_arrays, max_validation_samples
)
assert last_value_raw.shape[1:] == (5, 5)
print("LastValuePersistence PASS")

## 16 - Daily Seasonal Persistence

Horizon `h` dung state `t+h-24`, luon nam trong 24h input; `h=24` dung dung state tai `t`.

In [ ]:
assert seasonal_raw.shape == last_value_raw.shape
first_scenario, first_input_end = validation_sequence_index.resolve(0)
first_arrays = scenario_arrays[first_scenario]
for horizon_index, horizon in enumerate(FORECAST_HORIZONS):
    expected = first_arrays.raw_targets[first_input_end + horizon - 24]
    np.testing.assert_array_equal(seasonal_raw[0, horizon_index], expected)
np.testing.assert_array_equal(seasonal_raw[0, -1], first_arrays.raw_targets[first_input_end])
print("DailySeasonalPersistence PASS; no future information used")

## 17 - Model Definitions

Direct GRU/LSTM: hidden 64, one layer, Linear to 25 values, reshape `[B,5,5]`. No decoder/teacher forcing.

In [ ]:
@dataclass(frozen=True)
class MultiHorizonModelConfig:
    input_size: int = 8
    hidden_size: int = 64
    num_layers: int = 1
    num_horizons: int = 5
    output_size: int = 5


class DirectMultiHorizonGRUForecaster(nn.Module):
    def __init__(self, config: MultiHorizonModelConfig) -> None:
        super().__init__()
        self.config = config
        self.recurrent = nn.GRU(
            config.input_size, config.hidden_size, config.num_layers, batch_first=True
        )
        self.output_head = nn.Linear(
            config.hidden_size, config.num_horizons * config.output_size
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        _, hidden = self.recurrent(features)
        flat = self.output_head(hidden[-1])
        return flat.reshape(-1, self.config.num_horizons, self.config.output_size)


class DirectMultiHorizonLSTMForecaster(nn.Module):
    def __init__(self, config: MultiHorizonModelConfig) -> None:
        super().__init__()
        self.config = config
        self.recurrent = nn.LSTM(
            config.input_size, config.hidden_size, config.num_layers, batch_first=True
        )
        self.output_head = nn.Linear(
            config.hidden_size, config.num_horizons * config.output_size
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.recurrent(features)
        flat = self.output_head(hidden[-1])
        return flat.reshape(-1, self.config.num_horizons, self.config.output_size)


model_config = MultiHorizonModelConfig()
assert (model_config.hidden_size, model_config.num_layers) == (64, 1)

## 18 - Training Utilities

Shared MSE/AdamW loop, finite checks, gradient clipping, validation-only early stopping va atomic checkpoint.

In [ ]:
CHECKPOINT_DIR = MULTI_HORIZON_ARTIFACT_DIR / "checkpoints"
HISTORY_DIR = MULTI_HORIZON_ARTIFACT_DIR / "histories"
METRICS_DIR = MULTI_HORIZON_ARTIFACT_DIR / "metrics"
PLOT_DIR = MULTI_HORIZON_ARTIFACT_DIR / "plots"
for directory in (CHECKPOINT_DIR, HISTORY_DIR, METRICS_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def train_one_epoch(model, loader, optimizer, criterion, max_batches=None) -> float:
    model.train()
    total, count = 0.0, 0
    for batch_index, (features, targets) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        features = features.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        targets = targets.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        predictions = model(features)
        loss = criterion(predictions, targets)
        if not torch.isfinite(predictions).all() or not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training prediction/loss")
        loss.backward()
        for name, parameter in model.named_parameters():
            if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
                raise FloatingPointError(f"Non-finite gradient: {name}")
        nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()
        total += float(loss.detach()) * len(features)
        count += len(features)
    return total / count


@torch.no_grad()
def evaluate_loss(model, loader, criterion, max_batches=None) -> float:
    model.eval()
    total, count = 0.0, 0
    for batch_index, (features, targets) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        features = features.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        targets = targets.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        predictions = model(features)
        loss = criterion(predictions, targets)
        if not torch.isfinite(predictions).all() or not torch.isfinite(loss):
            raise FloatingPointError("Non-finite validation prediction/loss")
        total += float(loss) * len(features)
        count += len(features)
    return total / count


def save_checkpoint_atomic(path: Path, payload: dict[str, object]) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def load_checkpoint(path: Path, location):
    try:
        return torch.load(path, map_location=location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=location)


def fit_model(model_name, model, checkpoint_path) -> dict[str, object]:
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    epochs = SMOKE_MAX_EPOCHS if MULTI_HORIZON_SMOKE_TEST else MAX_EPOCHS
    train_limit = SMOKE_MAX_TRAIN_BATCHES if MULTI_HORIZON_SMOKE_TEST else None
    validation_limit = SMOKE_MAX_VALIDATION_BATCHES if MULTI_HORIZON_SMOKE_TEST else None
    history, best_loss, best_epoch, patience = [], math.inf, 0, 0
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, train_limit)
        validation_loss = evaluate_loss(model, validation_loader, criterion, validation_limit)
        history.append({
            "epoch": epoch, "train_loss": train_loss,
            "validation_loss": validation_loss,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_duration_seconds": time.perf_counter() - started,
        })
        if validation_loss < best_loss:
            best_loss, best_epoch, patience = validation_loss, epoch, 0
            save_checkpoint_atomic(checkpoint_path, {
                "model_name": model_name,
                "model_state_dict": model.state_dict(),
                "model_config": asdict(model_config),
                "forecast_horizons": list(FORECAST_HORIZONS),
                "target_columns": TARGET_COLUMNS,
                "feature_columns": FEATURE_COLUMNS,
                "lookback_steps": LOOKBACK_STEPS,
                "hidden_size": HIDDEN_SIZE,
                "num_layers": NUM_LAYERS,
                "seed": SEED,
                "epoch": epoch,
                "best_validation_loss": best_loss,
                "loss_definition": "equal-weight MSE over standardized [horizon,target] tensor",
                "training_mode": "direct_multi_horizon",
                "future_control_policy": "past_only",
            })
        else:
            patience += 1
        print(f"{model_name} epoch {epoch:02d}: train={train_loss:.6f}, val={validation_loss:.6f}")
        if patience >= EARLY_STOPPING_PATIENCE:
            break
    checkpoint = load_checkpoint(checkpoint_path, DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    return {
        "model": model, "history": history, "best_epoch": best_epoch,
        "best_validation_loss": best_loss, "checkpoint_path": checkpoint_path,
    }

## 19 - Train GRU

Train from scratch; one-step checkpoint khong duoc dung initialize.

In [ ]:
set_reproducibility(SEED)
data_loader_generator.manual_seed(SEED)
gru_model = DirectMultiHorizonGRUForecaster(model_config).to(DEVICE)
gru_result = fit_model(
    "DirectMultiHorizonGRU",
    gru_model,
    CHECKPOINT_DIR / "best_gru_multihorizon.pt",
)
gru_model = gru_result["model"]

## 20 - Train LSTM

Same capacity, seed, data, optimizer va aggregate loss de so sanh cong bang.

In [ ]:
set_reproducibility(SEED)
data_loader_generator.manual_seed(SEED)
lstm_model = DirectMultiHorizonLSTMForecaster(model_config).to(DEVICE)
lstm_result = fit_model(
    "DirectMultiHorizonLSTM",
    lstm_model,
    CHECKPOINT_DIR / "best_lstm_multihorizon.pt",
)
lstm_model = lstm_result["model"]

## 21 - Validation-Based Model Selection

Preferred recurrent baseline dung minimum aggregate validation standardized MSE tren 25 outputs.

In [ ]:
validation_selection = {
    "GRU": float(gru_result["best_validation_loss"]),
    "LSTM": float(lstm_result["best_validation_loss"]),
}
preferred_model_name = min(validation_selection, key=validation_selection.get)
preferred_model = gru_model if preferred_model_name == "GRU" else lstm_model
model_selection_record = {
    "criterion": "minimum aggregate validation standardized MSE over 5 horizons x 5 targets",
    "validation_losses": validation_selection,
    "preferred_model": preferred_model_name,
    "final_test_metrics_used": False,
}
print(f"Validation-only preferred recurrent model: {preferred_model_name}")

## 22 - Multi-Horizon Validation Predictions

Generate validation-only predictions for two persistence baselines and frozen GRU/LSTM.

In [ ]:
def transform_target_cube(raw_cube: np.ndarray) -> np.ndarray:
    original_shape = raw_cube.shape
    transformed = target_scaler.transform(raw_cube.reshape(-1, len(TARGET_COLUMNS)).astype(np.float64))
    return transformed.reshape(original_shape).astype(np.float32)


@torch.no_grad()
def predict_validation(model, max_batches=None) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    predictions, targets = [], []
    for batch_index, (features, batch_targets) in enumerate(validation_loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        predictions.append(model(features.to(DEVICE)).cpu().numpy())
        targets.append(batch_targets.numpy())
    return np.concatenate(predictions), np.concatenate(targets)


last_value_scaled = transform_target_cube(last_value_raw)
seasonal_scaled = transform_target_cube(seasonal_raw)
true_validation_scaled = transform_target_cube(validation_targets_raw)
validation_batch_limit = SMOKE_MAX_VALIDATION_BATCHES if MULTI_HORIZON_SMOKE_TEST else None
gru_scaled, neural_targets_scaled = predict_validation(gru_model, validation_batch_limit)
lstm_scaled, second_targets_scaled = predict_validation(lstm_model, validation_batch_limit)
np.testing.assert_allclose(neural_targets_scaled, second_targets_scaled, rtol=0, atol=0)
np.testing.assert_allclose(neural_targets_scaled, true_validation_scaled, rtol=2e-5, atol=2e-5)
validation_predictions_scaled = {
    "LastValuePersistence": last_value_scaled,
    "DailySeasonalPersistence": seasonal_scaled,
    "GRU": gru_scaled,
    "LSTM": lstm_scaled,
}
assert all(value.shape[1:] == (5, 5) for value in validation_predictions_scaled.values())

## 23 - Physical Metrics by Horizon

Inverse-transform va tinh MAE/RMSE/R2 tung target, standardized MSE tung horizon. Validation only.

In [ ]:
def inverse_target_cube(scaled_cube: np.ndarray) -> np.ndarray:
    shape = scaled_cube.shape
    return target_scaler.inverse_transform(
        scaled_cube.reshape(-1, len(TARGET_COLUMNS))
    ).reshape(shape)


def metrics_by_horizon(model_name, prediction_scaled, target_scaled) -> list[dict[str, object]]:
    prediction_raw = inverse_target_cube(prediction_scaled)
    target_raw = inverse_target_cube(target_scaled)
    rows = []
    for horizon_index, horizon in enumerate(FORECAST_HORIZONS):
        row = {
            "Model": model_name,
            "HorizonHours": horizon,
            "standardized_MSE": float(np.mean(
                (prediction_scaled[:, horizon_index] - target_scaled[:, horizon_index]) ** 2
            )),
        }
        for target_index, target_name in enumerate(TARGET_COLUMNS):
            truth = target_raw[:, horizon_index, target_index].astype(np.float64)
            prediction = prediction_raw[:, horizon_index, target_index].astype(np.float64)
            residual = truth - prediction
            denominator = float(np.sum((truth - truth.mean()) ** 2))
            r2 = 0.0 if denominator <= np.finfo(np.float64).eps else 1.0 - float(np.sum(residual**2)) / denominator
            row[f"{target_name}_MAE"] = float(np.mean(np.abs(residual)))
            row[f"{target_name}_RMSE"] = float(np.sqrt(np.mean(residual**2)))
            row[f"{target_name}_R2"] = r2
        rows.append(row)
    return rows


validation_metric_rows = [
    row
    for model_name, predictions in validation_predictions_scaled.items()
    for row in metrics_by_horizon(model_name, predictions, true_validation_scaled)
]
validation_by_horizon = pd.DataFrame(validation_metric_rows)
if not np.isfinite(validation_by_horizon.select_dtypes(include=[np.number]).to_numpy()).all():
    raise FloatingPointError("Validation metrics contain NaN/Inf")
print("Validation-only physical metrics by horizon PASS")

## 24 - Horizon Degradation

MAE/RMSE ratio relative to each model-target at h=1; khong average units khac nhau.

In [ ]:
degradation_rows = []
for model_name in validation_by_horizon["Model"].unique():
    model_rows = validation_by_horizon[validation_by_horizon["Model"] == model_name]
    reference = model_rows[model_rows["HorizonHours"] == 1].iloc[0]
    for _, row in model_rows.iterrows():
        for target_name in TARGET_COLUMNS:
            degradation_rows.append({
                "Model": model_name,
                "HorizonHours": int(row["HorizonHours"]),
                "Target": target_name,
                "MAE_ratio_vs_h1": float(row[f"{target_name}_MAE"] / reference[f"{target_name}_MAE"]),
                "RMSE_ratio_vs_h1": float(row[f"{target_name}_RMSE"] / reference[f"{target_name}_RMSE"]),
            })
horizon_degradation = pd.DataFrame(degradation_rows)
assert np.isfinite(horizon_degradation.select_dtypes(include=[np.number]).to_numpy()).all()

## 25 - Skill Scores

Per-horizon skill vs LastValue va DailySeasonal standardized MSE. Physical metrics van duoc giu rieng.

In [ ]:
skill_rows = []
for _, row in validation_by_horizon.iterrows():
    horizon = int(row["HorizonHours"])
    last_mse = float(validation_by_horizon[
        (validation_by_horizon["Model"] == "LastValuePersistence")
        & (validation_by_horizon["HorizonHours"] == horizon)
    ]["standardized_MSE"].iloc[0])
    seasonal_mse = float(validation_by_horizon[
        (validation_by_horizon["Model"] == "DailySeasonalPersistence")
        & (validation_by_horizon["HorizonHours"] == horizon)
    ]["standardized_MSE"].iloc[0])
    skill_rows.append({
        "Model": row["Model"],
        "HorizonHours": horizon,
        "Skill_vs_LastValue": 1.0 - float(row["standardized_MSE"]) / last_mse,
        "Skill_vs_Seasonal": 1.0 - float(row["standardized_MSE"]) / seasonal_mse,
    })
skill_scores = pd.DataFrame(skill_rows)
assert np.isfinite(skill_scores.select_dtypes(include=[np.number]).to_numpy()).all()

## 26 - Future Actuator Change Audit

Post-hoc labels mark any pump/fan/light change in `t+1..t+h`. Labels never enter model features.

In [ ]:
def future_control_change_labels(
    index: MultiHorizonIndex,
    arrays_by_scenario: dict[str, ScenarioArrays],
    sample_count: int,
) -> np.ndarray:
    labels = np.zeros((sample_count, len(FORECAST_HORIZONS)), dtype=bool)
    prefix_by_scenario = {}
    for scenario_id in index.scenario_ids:
        actuators = arrays_by_scenario[scenario_id].raw_actuators
        changes = np.zeros(len(actuators), dtype=np.int64)
        changes[1:] = np.any(actuators[1:] != actuators[:-1], axis=1).astype(np.int64)
        prefix_by_scenario[scenario_id] = np.cumsum(changes)
    for item in range(sample_count):
        scenario_id, input_end = index.resolve(item)
        prefix = prefix_by_scenario[scenario_id]
        for horizon_index, horizon in enumerate(FORECAST_HORIZONS):
            labels[item, horizon_index] = prefix[input_end + horizon] - prefix[input_end] > 0
    return labels


control_change_labels = future_control_change_labels(
    validation_sequence_index, scenario_arrays, len(true_validation_scaled)
)
preferred_predictions = validation_predictions_scaled[preferred_model_name]
control_audit_rows = []
for horizon_index, horizon in enumerate(FORECAST_HORIZONS):
    labels = control_change_labels[:, horizon_index]
    for change_value in (False, True):
        mask = labels == change_value
        if not mask.any():
            continue
        residual = preferred_predictions[mask, horizon_index] - true_validation_scaled[mask, horizon_index]
        row = {
            "Model": preferred_model_name,
            "HorizonHours": horizon,
            "future_control_change": bool(change_value),
            "window_count": int(mask.sum()),
            "window_fraction": float(mask.mean()),
            "aggregate_standardized_MSE": float(np.mean(residual**2)),
        }
        raw_prediction = inverse_target_cube(preferred_predictions[mask])[:, horizon_index]
        raw_truth = inverse_target_cube(true_validation_scaled[mask])[:, horizon_index]
        for target_index, target_name in enumerate(TARGET_COLUMNS):
            row[f"{target_name}_RMSE"] = float(np.sqrt(np.mean(
                (raw_prediction[:, target_index] - raw_truth[:, target_index]) ** 2
            )))
        control_audit_rows.append(row)
future_control_audit = pd.DataFrame(control_audit_rows)
assert np.isfinite(future_control_audit.select_dtypes(include=[np.number]).to_numpy()).all()

## 27 - Diagnostic Plots

Loss curves, MSE/MAE vs horizon va deterministic contiguous 24-hour-ahead validation segment.

In [ ]:
def plot_loss(history, model_name, path):
    epochs = [record["epoch"] for record in history]
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, [record["train_loss"] for record in history], label="Train")
    plt.plot(epochs, [record["validation_loss"] for record in history], label="Validation")
    plt.xlabel("Epoch"); plt.ylabel("Standardized MSE"); plt.title(model_name); plt.legend(); plt.grid(alpha=.25)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


plot_loss(gru_result["history"], "Direct Multi-Horizon GRU", PLOT_DIR / "gru_loss_curve.png")
plot_loss(lstm_result["history"], "Direct Multi-Horizon LSTM", PLOT_DIR / "lstm_loss_curve.png")

plt.figure(figsize=(8, 5))
for model_name, rows in validation_by_horizon.groupby("Model"):
    plt.plot(rows["HorizonHours"], rows["standardized_MSE"], marker="o", label=model_name)
plt.xlabel("Forecast horizon (hours)"); plt.ylabel("Standardized MSE"); plt.legend(); plt.grid(alpha=.25)
plt.tight_layout(); plt.savefig(PLOT_DIR / "standardized_mse_vs_horizon.png", dpi=150); plt.close()

figure, axes = plt.subplots(3, 2, figsize=(12, 12))
for axis, target_name in zip(axes.flat, TARGET_COLUMNS):
    for model_name, rows in validation_by_horizon.groupby("Model"):
        axis.plot(rows["HorizonHours"], rows[f"{target_name}_MAE"], marker="o", label=model_name)
    axis.set_title(target_name); axis.set_xlabel("Horizon (h)"); axis.set_ylabel("MAE"); axis.grid(alpha=.2)
axes.flat[-1].axis("off"); axes.flat[0].legend(fontsize=8)
figure.tight_layout(); figure.savefig(PLOT_DIR / "mae_vs_horizon_by_target.png", dpi=150); plt.close(figure)

horizon_24_index = FORECAST_HORIZONS.index(24)
representative_count = min(72, len(validation_sequence_index))
representative_predictions = inverse_target_cube(
    validation_predictions_scaled[preferred_model_name][:representative_count]
)[:, horizon_24_index]
representative_truth = inverse_target_cube(
    true_validation_scaled[:representative_count]
)[:, horizon_24_index]
representative_codes = validation_sequence_index.scenario_codes[:representative_count]
if not (representative_codes == representative_codes[0]).all():
    raise AssertionError("Representative plot crosses scenario boundary")
representative_id = validation_sequence_index.scenario_ids[int(representative_codes[0])]
representative_input_ends = validation_sequence_index.input_end_positions[:representative_count]
representative_timestamps = scenario_arrays[representative_id].timestamps[
    representative_input_ends + 24
]
figure, axes = plt.subplots(5, 1, figsize=(12, 14), sharex=True)
for target_index, (axis, target_name) in enumerate(zip(axes, TARGET_COLUMNS)):
    axis.plot(representative_timestamps, representative_truth[:, target_index], label="Actual")
    axis.plot(representative_timestamps, representative_predictions[:, target_index], label="Predicted")
    axis.set_ylabel(target_name); axis.grid(alpha=.2)
axes[0].legend(); axes[-1].set_xlabel("Target timestamp")
figure.suptitle(f"{preferred_model_name} 24-hour-ahead forecast (input ends 24h earlier)")
figure.tight_layout(); figure.savefig(PLOT_DIR / "preferred_model_24h_validation.png", dpi=150); plt.close(figure)

## 28 - Save Artifacts

New directory only; Notebook 02 artifacts are never overwritten. Smoke artifacts remain isolated.

In [ ]:
def write_json(path: Path, payload: object) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


write_json(HISTORY_DIR / "gru_multihorizon_history.json", gru_result["history"])
write_json(HISTORY_DIR / "lstm_multihorizon_history.json", lstm_result["history"])
write_json(METRICS_DIR / "validation_metrics.json", validation_metric_rows)
validation_by_horizon.to_csv(METRICS_DIR / "validation_by_horizon.csv", index=False)
horizon_degradation.to_csv(METRICS_DIR / "horizon_degradation.csv", index=False)
skill_scores.to_csv(METRICS_DIR / "skill_scores.csv", index=False)
future_control_audit.to_csv(METRICS_DIR / "future_control_audit.csv", index=False)
multi_horizon_run_manifest = {
    "seed": SEED,
    "training_mode": "direct_multi_horizon",
    "multi_horizon_smoke_test": MULTI_HORIZON_SMOKE_TEST,
    "lookback_steps": LOOKBACK_STEPS,
    "forecast_horizons": list(FORECAST_HORIZONS),
    "feature_columns": FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "future_control_policy": "past_only_diagnostic_after_prediction",
    "model_config": asdict(model_config),
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "development_scenario_ids": development_scenario_ids,
    "held_out_scenario_ids_provenance_only": held_out_scenario_ids,
    "train_range": [str(train_start), str(train_end)],
    "validation_range": [str(validation_start), str(validation_end)],
    "expected_full_window_counts": {
        "train": EXPECTED_TRAIN_WINDOWS, "validation": EXPECTED_VALIDATION_WINDOWS
    },
    "actual_execution_window_counts": actual_window_counts,
    "model_selection": model_selection_record,
    "multi_horizon_final_tests_executed": False,
}
write_json(MULTI_HORIZON_ARTIFACT_DIR / "multi_horizon_run_manifest.json", multi_horizon_run_manifest)
print(f"Artifacts saved to {MULTI_HORIZON_ARTIFACT_DIR.resolve()}")

## 29 - Checkpoint Reload Verification

Fresh instances phai reproduce deterministic validation predictions `[B,5,5]`.

In [ ]:
deterministic_features = next(iter(validation_loader))[0].to(DEVICE)


@torch.no_grad()
def verify_reload(model_name, trained_model, path):
    trained_model.eval()
    reference = trained_model(deterministic_features).cpu()
    checkpoint = load_checkpoint(path, "cpu")
    config = MultiHorizonModelConfig(**checkpoint["model_config"])
    if model_name == "GRU":
        fresh = DirectMultiHorizonGRUForecaster(config)
    elif model_name == "LSTM":
        fresh = DirectMultiHorizonLSTMForecaster(config)
    else:
        raise ValueError(model_name)
    fresh.load_state_dict(checkpoint["model_state_dict"])
    fresh = fresh.to(DEVICE).eval()
    reloaded = fresh(deterministic_features).cpu()
    assert reloaded.shape[1:] == (5, 5)
    torch.testing.assert_close(reference, reloaded, rtol=1e-6, atol=1e-7)
    if checkpoint["forecast_horizons"] != list(FORECAST_HORIZONS):
        raise ValueError("Checkpoint horizon ordering mismatch")
    return {"status": "PASS", "shape": list(reloaded.shape), "checkpoint": str(path)}


checkpoint_reload_audit = {
    "GRU": verify_reload("GRU", gru_model, CHECKPOINT_DIR / "best_gru_multihorizon.pt"),
    "LSTM": verify_reload("LSTM", lstm_model, CHECKPOINT_DIR / "best_lstm_multihorizon.pt"),
}
print(f"Checkpoint reload PASS: {checkpoint_reload_audit}")

## 30 - Final Summary

Machine-checkable gate. Smoke values are not scientific full-training results; final tests remain unopened.

In [ ]:
experiment_summary = {
    "status": "PASS",
    "canonical_scenarios": len(canonical_index),
    "development_scenarios": len(development_scenario_ids),
    "held_out_scenarios": len(held_out_scenario_ids),
    "lookback_steps": LOOKBACK_STEPS,
    "forecast_horizons": list(FORECAST_HORIZONS),
    "input_shape": ["B", 24, 8],
    "target_shape": ["B", 5, 5],
    "expected_full_window_counts": {
        "train": EXPECTED_TRAIN_WINDOWS, "validation": EXPECTED_VALIDATION_WINDOWS
    },
    "actual_execution_window_counts": actual_window_counts,
    "scaler_refit_performed": False,
    "held_out_csv_loaded": False,
    "final_test_loaders_constructed": False,
    "multi_horizon_final_tests_executed": False,
    "future_control_policy": "past_only; future controls diagnostic only",
    "last_value_persistence": "PASS",
    "daily_seasonal_persistence": "PASS",
    "gru_forward_backward": "PASS",
    "lstm_forward_backward": "PASS",
    "checkpoint_reload": checkpoint_reload_audit,
    "horizon_metrics": "PASS",
    "horizon_degradation": "PASS",
    "skill_scores": "PASS",
    "future_control_audit": "PASS",
    "multi_horizon_smoke_test": MULTI_HORIZON_SMOKE_TEST,
    "full_multi_horizon_training_executed": not MULTI_HORIZON_SMOKE_TEST,
}
print(json.dumps(experiment_summary, indent=2))